In [1]:
# ------------------------------------------------------------
# Imports
# ------------------------------------------------------------
from pathlib import Path
import numpy as np
import pandas as pd
import yaml
import torch

import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans
from sklearn.metrics import (
    adjusted_mutual_info_score,
    adjusted_rand_score,
    homogeneity_score,
)
from sklearn.neighbors import NearestNeighbors
import umap
import igraph as ig
import leidenalg as la
import joblib

from clearit.config import MODELS_DIR, DATASETS_DIR, OUTPUTS_DIR
from clearit.inference.utils import get_embeddings

# ------------------------------------------------------------
# Configuration
# ------------------------------------------------------------
encoder_id = "E0030"                     # pretrained encoder

# Train side: from encoder config (TME-A_ML6)
# Test side: explicit inForm_MC7 split
test_annotation_name = "inForm_MC7"
test_index_rel_path  = "indices/test/test_full01.npz"

n_train_subsample = 200_000
n_test_subsample  = 50_000

pca_components       = 50
kmeans_clusters      = 10
knn_neighbors_graph  = 30
knn_neighbors_assign = 10

umap_n_neighbors = 15
umap_min_dist    = 0.10

output_dir = (
    OUTPUTS_DIR
    / "unsupervised_clustering"
    / "TNBC1-MxIF8"
    / test_annotation_name
    / encoder_id
)
output_dir.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Load encoder config (defines train dataset / annotation / indices)
# ------------------------------------------------------------
enc_dir = MODELS_DIR / "encoders" / encoder_id
enc_cfg_path = enc_dir / "conf_enc.yaml"
enc_cfg = yaml.safe_load(enc_cfg_path.read_text())

train_dataset_name    = enc_cfg["dataset_name"]       # e.g. "TNBC1-MxIF8"
train_annotation_name = enc_cfg["annotation_name"]    # e.g. "TME-A_ML6"
train_index_rel_path  = enc_cfg["data_index_list"]    # e.g. "indices/train/train_full01.npz"

# ------------------------------------------------------------
# Build TRAIN dataframe (TME-A_ML6, from encoder pretraining split)
# ------------------------------------------------------------
train_labels_path = DATASETS_DIR / train_dataset_name / train_annotation_name / "labels.csv"
df_train_all = pd.read_csv(train_labels_path)

train_npz_path = DATASETS_DIR / train_dataset_name / train_annotation_name / train_index_rel_path
arr_train = np.load(train_npz_path)
train_files = arr_train.files

if len(train_files) == 1:
    train_indices = arr_train[train_files[0]]
else:
    # encoder pretraining NPZ may contain indices + channels;
    # for clustering we only need the indices
    train_indices = arr_train[train_files[0]]

df_train = df_train_all.iloc[train_indices].reset_index(drop=True)

key_cols = ["fname", "cell_x", "cell_y"]
df_train = df_train.drop_duplicates(subset=key_cols).reset_index(drop=True)

# ------------------------------------------------------------
# Build TEST dataframe (inForm_MC7)
# ------------------------------------------------------------
dataset_name = train_dataset_name  # same images as for TME-A_ML6

test_labels_path = DATASETS_DIR / dataset_name / test_annotation_name / "labels.csv"
df_test_all = pd.read_csv(test_labels_path)

test_npz_path = DATASETS_DIR / dataset_name / test_annotation_name / test_index_rel_path
arr_test = np.load(test_npz_path)
test_files = arr_test.files
test_indices = arr_test[test_files[0]]

df_test = df_test_all.iloc[test_indices].reset_index(drop=True)
df_test = df_test.drop_duplicates(subset=key_cols).reset_index(drop=True)

# ------------------------------------------------------------
# Subsample train / test for feasibility
# ------------------------------------------------------------
rng = np.random.default_rng(0)

if len(df_train) > n_train_subsample:
    df_train = df_train.sample(n_train_subsample, random_state=0).reset_index(drop=True)

if len(df_test) > n_test_subsample:
    df_test = df_test.sample(n_test_subsample, random_state=0).reset_index(drop=True)

# ensure encoder uses correct patch size if it is stored
if "img_size" in enc_cfg:
    df_train.attrs["img_size"] = int(enc_cfg["img_size"])
    df_test.attrs["img_size"]  = int(enc_cfg["img_size"])

# ------------------------------------------------------------
# Extract encoder embeddings (4096-D multiplex features)
# ------------------------------------------------------------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

X_train = get_embeddings(
    df_train,
    dataset_name=train_dataset_name,
    annotation_name=train_annotation_name,
    encoder_id=encoder_id,
    batch_size=128,
    num_workers=4,
    proj_layers=0,
    device=device,
).cpu().numpy()

X_test = get_embeddings(
    df_test,
    dataset_name=dataset_name,
    annotation_name=test_annotation_name,
    encoder_id=encoder_id,
    batch_size=128,
    num_workers=4,
    proj_layers=0,
    device=device,
).cpu().numpy()

# inForm_MC7 labels (for evaluation only)
if "label" not in df_test.columns:
    raise KeyError("Expected an integer 'label' column in MC7 labels.")
y_test = df_test["label"].astype(int).to_numpy()

# ------------------------------------------------------------
# PCA (fit on TME train, apply to MC7 test)
# ------------------------------------------------------------
pca_dim = min(pca_components, X_train.shape[1])
pca = PCA(n_components=pca_dim, random_state=0)

X_train_pca = pca.fit_transform(X_train)
X_test_pca  = pca.transform(X_test)

joblib.dump(pca, output_dir / "pca_model.joblib")

# ------------------------------------------------------------
# k-means clustering (unsupervised)
# ------------------------------------------------------------
kmeans = KMeans(n_clusters=kmeans_clusters, random_state=0, n_init=10)
cluster_train_km = kmeans.fit_predict(X_train_pca)
cluster_test_km  = kmeans.predict(X_test_pca)

joblib.dump(kmeans, output_dir / "kmeans_model.joblib")

# ------------------------------------------------------------
# Leiden clustering (graph-based on train, kNN assignment for test)
# ------------------------------------------------------------
nn_graph = NearestNeighbors(n_neighbors=knn_neighbors_graph + 1, metric="euclidean")
nn_graph.fit(X_train_pca)
distances, indices = nn_graph.kneighbors(X_train_pca)

sources = np.repeat(np.arange(X_train_pca.shape[0]), knn_neighbors_graph)
targets = indices[:, 1:].reshape(-1)

g = ig.Graph(n=X_train_pca.shape[0], edges=list(zip(sources, targets)), directed=False)
g.simplify(combine_edges="first")

partition = la.find_partition(
    g,
    la.RBConfigurationVertexPartition,
    resolution_parameter=1.0,
)
cluster_train_leiden = np.array(partition.membership, dtype=int)

# assign test clusters by kNN in PCA space
nn_assign = NearestNeighbors(n_neighbors=knn_neighbors_assign, metric="euclidean")
nn_assign.fit(X_train_pca)
dist_test, idx_test = nn_assign.kneighbors(X_test_pca)

cluster_test_leiden = []
for neigh in idx_test:
    labs = cluster_train_leiden[neigh]
    vals, counts = np.unique(labs, return_counts=True)
    cluster_test_leiden.append(vals[counts.argmax()])
cluster_test_leiden = np.array(cluster_test_leiden, dtype=int)

# save cluster assignments
np.savez_compressed(
    output_dir / "clusters_train.npz",
    kmeans=cluster_train_km,
    leiden=cluster_train_leiden,
)

np.savez_compressed(
    output_dir / "clusters_test.npz",
    kmeans=cluster_test_km,
    leiden=cluster_test_leiden,
    y_test=y_test,
)

# ------------------------------------------------------------
# Metrics vs MC7 labels (diagnostic only)
# ------------------------------------------------------------
metrics = {}

ami_km  = adjusted_mutual_info_score(y_test, cluster_test_km)
ari_km  = adjusted_rand_score(y_test, cluster_test_km)
homo_km = homogeneity_score(y_test, cluster_test_km)
metrics["kmeans"] = {
    "AMI": float(ami_km),
    "ARI": float(ari_km),
    "Homogeneity": float(homo_km),
}

ami_lei  = adjusted_mutual_info_score(y_test, cluster_test_leiden)
ari_lei  = adjusted_rand_score(y_test, cluster_test_leiden)
homo_lei = homogeneity_score(y_test, cluster_test_leiden)
metrics["leiden"] = {
    "AMI": float(ami_lei),
    "ARI": float(ari_lei),
    "Homogeneity": float(homo_lei),
}

with open(output_dir / "metrics.yaml", "w") as f:
    yaml.safe_dump(metrics, f)

print("Clustering metrics (test vs MC7):")
for name, m in metrics.items():
    print(
        f"{name:7s} → AMI={m['AMI']:.3f}, "
        f"ARI={m['ARI']:.3f}, "
        f"Homo={m['Homogeneity']:.3f}"
    )

# ------------------------------------------------------------
# UMAP on PCA(test) for visualization
# ------------------------------------------------------------
umap_model = umap.UMAP(
    n_neighbors=umap_n_neighbors,
    min_dist=umap_min_dist,
    metric="euclidean",
    random_state=0,
)

emb_test = umap_model.fit_transform(X_test_pca)
np.savez_compressed(output_dir / "umap_test.npz", emb_test=emb_test)
joblib.dump(umap_model, output_dir / "umap_model.joblib")

# ------------------------------------------------------------
# Plots: UMAP colored by MC7 label, k-means cluster, Leiden cluster
# ------------------------------------------------------------
plt.figure(figsize=(6, 6))
plt.scatter(
    emb_test[:, 0],
    emb_test[:, 1],
    c=y_test,
    cmap="tab20",
    s=3,
    alpha=0.7,
)
plt.title("UMAP (test, inForm_MC7) – colored by MC7 label")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.tight_layout()
plt.savefig(output_dir / "umap_mc7_labels.png", dpi=300)
plt.close()

plt.figure(figsize=(6, 6))
plt.scatter(
    emb_test[:, 0],
    emb_test[:, 1],
    c=cluster_test_km,
    cmap="tab20",
    s=3,
    alpha=0.7,
)
plt.title("UMAP (test) – colored by k-means clusters")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.tight_layout()
plt.savefig(output_dir / "umap_kmeans_clusters.png", dpi=300)
plt.close()

plt.figure(figsize=(6, 6))
plt.scatter(
    emb_test[:, 0],
    emb_test[:, 1],
    c=cluster_test_leiden,
    cmap="tab20",
    s=3,
    alpha=0.7,
)
plt.title("UMAP (test) – colored by Leiden clusters")
plt.xlabel("UMAP-1")
plt.ylabel("UMAP-2")
plt.tight_layout()
plt.savefig(output_dir / "umap_leiden_clusters.png", dpi=300)
plt.close()

# ------------------------------------------------------------
# Save metadata for reproducibility
# ------------------------------------------------------------
metadata = {
    "encoder_id": encoder_id,
    "train_dataset_name": train_dataset_name,
    "train_annotation_name": train_annotation_name,
    "train_index_rel_path": train_index_rel_path,
    "test_dataset_name": dataset_name,
    "test_annotation_name": test_annotation_name,
    "test_index_rel_path": test_index_rel_path,
    "n_train_subsample": int(len(df_train)),
    "n_test_subsample": int(len(df_test)),
    "feature_dim": int(X_train.shape[1]),
    "pca_components": int(pca_dim),
    "kmeans_clusters": int(kmeans_clusters),
    "knn_neighbors_graph": int(knn_neighbors_graph),
    "knn_neighbors_assign": int(knn_neighbors_assign),
    "umap_n_neighbors": int(umap_n_neighbors),
    "umap_min_dist": float(umap_min_dist),
    "metrics": metrics,
}

with open(output_dir / "metadata.yaml", "w") as f:
    yaml.safe_dump(metadata, f)

print("All models, clusters, UMAP coordinates, metrics and plots saved to:")
print(output_dir)


Loaded & padded 761 images in 257.8s (3.0 f/s)
Extracted 200000 crops in 56.6s (3530.6 crops/s)
Loaded & padded 248 images in 8.6s (28.9 f/s)
Extracted 50000 crops in 13.8s (3610.3 crops/s)
Clustering metrics (test vs MC7):
kmeans  → AMI=0.000, ARI=0.000, Homo=0.001
leiden  → AMI=0.000, ARI=-0.000, Homo=0.001


OMP: Info #273: omp_set_nested routine deprecated, please use omp_set_max_active_levels instead.


All models, clusters, UMAP coordinates, metrics and plots saved to:
/workspace/files/CLEAR-IT/outputs/unsupervised_clustering/TNBC1-MxIF8/inForm_MC7/E0030
